In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Define save path for 1st model (update as per model)
model_save_path = "/content/drive/MyDrive/BanglaFakeReview/alpaca-orca-3b-instruct"

In [ ]:
!pip install -q transformers datasets accelerate bitsandbytes peft huggingface_hub torch==2.6.0 unsloth unsloth_zoo

In [ ]:
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
from datasets import load_dataset

dataset = load_dataset("shawon95/Bengali-Fake-Review-Dataset")
if "test" not in dataset:
    dataset = dataset["train"].train_test_split(test_size=0.2, seed=42)
print("✅ Dataset Loaded:", dataset)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/3.14k [00:00<?, ?B/s]

fake.csv:   0%|          | 0.00/1.90M [00:00<?, ?B/s]

non-fake.csv:   0%|          | 0.00/12.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9049 [00:00<?, ? examples/s]

✅ Dataset Loaded: DatasetDict({
    train: Dataset({
        features: ['Review', 'Label'],
        num_rows: 7239
    })
    test: Dataset({
        features: ['Review', 'Label'],
        num_rows: 1810
    })
})


In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "BanglaLLM/BanglaLLama-3.2-3b-bangla-alpaca-orca-instruct-v0.0.1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    tokens = tokenizer(examples["Review"], padding="max_length", truncation=True, max_length=256)
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_train = dataset["train"].map(tokenize_function, batched=True)
tokenized_test = dataset["test"].map(tokenize_function, batched=True)

tokenized_train = tokenized_train.remove_columns(["Review", "Label"])
tokenized_test = tokenized_test.remove_columns(["Review", "Label"])


tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/449 [00:00<?, ?B/s]

Map:   0%|          | 0/7239 [00:00<?, ? examples/s]

Map:   0%|          | 0/1810 [00:00<?, ? examples/s]

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=512,
    dtype=torch.float16,
    load_in_4bit=True
)

tokenizer.pad_token = tokenizer.eos_token

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing=True,
    random_state=42,
    use_rslora=False,
    loftq_config=None
)

model.print_trainable_parameters()


<ipython-input-7-8e80f8581234>:1: UserWarning: WARNING: Unsloth should be imported before transformers to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.5.9: Fast Llama patching. Transformers: 4.52.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/21.0k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.25G [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/449 [00:00<?, ?B/s]

BanglaLLM/BanglaLLama-3.2-3b-bangla-alpaca-orca-instruct-v0.0.1 does not have a padding token! Will use pad_token = <|finetune_right_pad_id|>.


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2025.5.9 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir=model_save_path,
    save_strategy="epoch",
    learning_rate=2e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,
    num_train_epochs=2,
    weight_decay=0.01,
    save_total_limit=2,
    logging_steps=10,
    fp16=True,
    report_to="none",
    remove_unused_columns=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=None
)

trainer.train()


<ipython-input-8-8f8b918d3178>:19: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,239 | Num Epochs = 2 | Total steps = 454
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 8 x 1) = 32
 "-____-"     Trainable parameters = 24,313,856/3,000,000,000 (0.81% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,1.487400
20,1.364900
30,1.319300
40,1.222200
50,1.190300
60,1.178900
70,1.197100
80,1.181900
90,1.154300
100,1.154800


TrainOutput(global_step=454, training_loss=1.0907095612933457, metrics={'train_runtime': 6652.0819, 'train_samples_per_second': 2.176, 'train_steps_per_second': 0.068, 'total_flos': 6.322458738976358e+16, 'train_loss': 1.0907095612933457, 'epoch': 2.0})

In [ ]:
import torch
from tqdm import tqdm

print("📝 Generating predictions using prompts...")

model.eval()
pred_labels = []
true_labels = []
review_texts = []

positive_keywords = ["ইতিবাচক", "positive", "ভালো", "ভাল", "great", "good", "অসাধারণ", "সেরা"]
negative_keywords = ["নেতিবাচক", "negative", "খারাপ", "বাজে", "poor", "bad", "terrible", "worst"]

prompt_template = "এই রিভিউটি কি ভুয়া নাকি আসল? রিভিউ:\n\"{review}\"\n\nউত্তর (ভুয়া বা আসল):"

for example in tqdm(dataset["test"], desc="Prompt-based Prediction"):
    review = example["Review"]
    true_label = example["Label"]

    # 📝 Create the prompt
    prompt = prompt_template.format(review=review)

    # Tokenize the prompt
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    )
    inputs = {k: v.to("cuda") for k, v in inputs.items()}

    # Generate the prediction
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=20,
            do_sample=False,
            num_beams=1,
            early_stopping=True
        )

    decoded = tokenizer.decode(output[0], skip_special_tokens=True).lower()

    # Simple keyword matching to determine label
    if any(word in decoded for word in negative_keywords):
        pred = 0  # Fake
    elif any(word in decoded for word in positive_keywords):
        pred = 1  # Non-Fake
    else:
        pred = 1  # Default to Non-Fake

    pred_labels.append(pred)
    true_labels.append(true_label)
    review_texts.append(review)

print("\n🔍 Prompt-based prediction complete.")



📝 Generating predictions using prompts...


Prompt-based Prediction: 100%|██████████| 1810/1810 [39:03<00:00,  1.30s/it]


🔍 Prompt-based prediction complete.


In [ ]:
import pandas as pd
import os

# Create DataFrame from the lists
df_pred = pd.DataFrame({
    "review": review_texts,
    "true_label": true_labels,
    "predicted_label": pred_labels
})

# Define save directory and path for Alpaca-Orca model
save_dir = "/content/drive/MyDrive/BanglaFakeReview/model_3_BanglaLLama_AlpacaOrca"
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

save_path = os.path.join(save_dir, "prompt_based_predictions.csv")

# Save CSV
df_pred.to_csv(save_path, index=False)
print(f"✅ Prompt-based predictions saved at: {save_path}")


✅ Prompt-based predictions saved at: /content/drive/MyDrive/BanglaFakeReview/model_3_BanglaLLama_AlpacaOrca/prompt_based_predictions.csv


In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

print("📊 Running Evaluation on Prompt-Based Predictions...")

true_labels = df_pred["true_label"].tolist()
pred_labels = df_pred["predicted_label"].tolist()

accuracy = accuracy_score(true_labels, pred_labels)
precision, recall, f1, _ = precision_recall_fscore_support(
    true_labels, pred_labels, average='weighted', zero_division=0
)

print(f"\n🔹 Accuracy: {accuracy:.4f}")
print(f"🔹 Precision: {precision:.4f}")
print(f"🔹 Recall: {recall:.4f}")
print(f"🔹 F1-Score: {f1:.4f}")

print("\n🔎 Detailed Classification Report:")
print(classification_report(true_labels, pred_labels, target_names=["Fake", "Non-Fake"], digits=4, zero_division=0))


📊 Running Evaluation on Prompt-Based Predictions...

🔹 Accuracy: 0.7890
🔹 Precision: 0.7334
🔹 Recall: 0.7890
🔹 F1-Score: 0.7577

🔎 Detailed Classification Report:
              precision    recall  f1-score   support

        Fake     0.1439    0.0707    0.0948       283
    Non-Fake     0.8426    0.9221    0.8806      1527

    accuracy                         0.7890      1810
   macro avg     0.4932    0.4964    0.4877      1810
weighted avg     0.7334    0.7890    0.7577      1810

